In [ ]:
import requests
import pandas as pd

def coletar_dados(codigo, nome_indicador):
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados?formato=json"

    response = requests.get(url)
    data = response.json()

    df = pd.DataFrame(data)

    df["indicador"] = nome_indicador

    return df


series = {
    "concessoes": 20633,
    "saldo": 22050,
    "inadimplencia": 21112,
    "endividamento": 29037
}

dfs = []

for nome, codigo in series.items():
    df = coletar_dados(codigo, nome)
    dfs.append(df)

df_final = pd.concat(dfs)

df_final["data"] = pd.to_datetime(df_final["data"], dayfirst=True)

df_final["valor"] = pd.to_numeric(df_final["valor"], errors="coerce")

df_final = df_final.dropna()

df_final = df_final.sort_values(by="data")

print(df_final)

          data       valor      indicador
0   2005-01-01       16.51  endividamento
1   2005-02-01       16.84  endividamento
2   2005-03-01       17.20  endividamento
3   2005-04-01       17.60  endividamento
4   2005-05-01       17.96  endividamento
..         ...         ...            ...
177 2025-12-01        6.86  inadimplencia
251 2025-12-01       49.69  endividamento
178 2026-01-01        6.95  inadimplencia
168 2026-01-01  4460716.00          saldo
178 2026-01-01   365636.00     concessoes

[779 rows x 3 columns]


In [ ]:
import sqlite3

conn = sqlite3.connect("credito.db")

df_final.to_sql("dados_credito", conn, if_exists="replace", index=False)

779

In [ ]:
!pip install fastapi uvicorn pyngrok nest-asyncio

In [ ]:
from fastapi import FastAPI
import sqlite3

app = FastAPI()

# conexão com banco
def get_connection():
    conn = sqlite3.connect("credito.db")
    conn.row_factory = sqlite3.Row
    return conn

# rota teste
@app.get("/")
def home():
    return {"mensagem": "API rodando 🚀"}

# todos os dados
@app.get("/dados")
def get_dados():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM dados_credito")
    rows = cursor.fetchall()

    conn.close()

    return [dict(row) for row in rows]

# filtro por indicador
@app.get("/dados/indicador/{indicador}")
def get_por_indicador(indicador: str):
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute(
        "SELECT * FROM dados_credito WHERE indicador = ?",
        (indicador,)
    )

    rows = cursor.fetchall()
    conn.close()

    return [dict(row) for row in rows]

# filtro por período
@app.get("/dados/periodo/")
def get_por_periodo(data_inicio: str, data_fim: str):
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute(
        """
        SELECT * FROM dados_credito
        WHERE data BETWEEN ? AND ?
        """,
        (data_inicio, data_fim)
    )

    rows = cursor.fetchall()
    conn.close()

    return [dict(row) for row in rows]

In [ ]:
import nest_asyncio
from threading import Thread
import uvicorn

nest_asyncio.apply()

def run():
    uvicorn.run(app, host="127.0.0.1", port=8000)

thread = Thread(target=run)
thread.start()

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3BLa8PEqeR8P7lSB6q2CcCFzO0T_7rMttMcPCjugrDH6nVpfk")

INFO:     Started server process [2779]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


In [ ]:
from pyngrok import ngrok

# abre um túnel público
public_url = ngrok.connect(8000)

print("URL pública:", public_url)

URL pública: NgrokTunnel: "https://bardier-lionel-pactionally.ngrok-free.dev" -> "http://localhost:8000"
